In [1]:
import os
import numpy as np
import tensorflow as tf
from keras.datasets import mnist
from keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

RUTA_MODELO = "numeros_conv_ad_do.h5"  # nombre del archivo del modelo


def cargar_datos_mnist():
    (X_entrenamiento, Y_entrenamiento), (X_pruebas, Y_pruebas) = mnist.load_data()

    # Colocar en forma (n, 28, 28, 1)
    X_entrenamiento = X_entrenamiento.reshape(-1, 28, 28, 1).astype("float32") / 255.0
    X_pruebas = X_pruebas.reshape(-1, 28, 28, 1).astype("float32") / 255.0

    # One-hot encoding
    Y_entrenamiento = to_categorical(Y_entrenamiento, 10)
    Y_pruebas = to_categorical(Y_pruebas, 10)

    return (X_entrenamiento, Y_entrenamiento), (X_pruebas, Y_pruebas)


def crear_modelo():
    modelo = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D(2, 2),

        tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(2, 2),

        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(100, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

    modelo.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modelo


def entrenar_y_guardar_modelo(ruta_modelo: str = RUTA_MODELO, epocas: int = 60):
    print("Cargando datos MNIST...")
    (X_entrenamiento, Y_entrenamiento), (X_pruebas, Y_pruebas) = cargar_datos_mnist()

    print("Configurando aumento de datos...")
    datagen = ImageDataGenerator(
        rotation_range=30,
        width_shift_range=0.25,
        height_shift_range=0.25,
        zoom_range=[0.5, 1.5],
    )
    datagen.fit(X_entrenamiento)

    print("Creando modelo...")
    modelo = crear_modelo()

    TAMANO_LOTE = 32
    steps_por_epoca = int(np.ceil(X_entrenamiento.shape[0] / float(TAMANO_LOTE)))
    steps_val = int(np.ceil(X_pruebas.shape[0] / float(TAMANO_LOTE)))

    print("Entrenando modelo...")
    history = modelo.fit(
        datagen.flow(X_entrenamiento, Y_entrenamiento, batch_size=TAMANO_LOTE),
        epochs=epocas,
        validation_data=(X_pruebas, Y_pruebas),
        steps_per_epoch=steps_por_epoca,
        validation_steps=steps_val
    )

    print(f"Guardando modelo en {ruta_modelo} ...")
    modelo.save(ruta_modelo)
    print("Modelo guardado correctamente.")

    return modelo, history


def cargar_o_entrenar_modelo(ruta_modelo: str = RUTA_MODELO, reentrenar: bool = False):
    """
    - Si el archivo existe y reentrenar = False -> solo lo carga.
    - Si no existe o reentrenar = True -> entrena y lo guarda.
    """
    if os.path.exists(ruta_modelo) and not reentrenar:
        print(f"Cargando modelo ya entrenado desde {ruta_modelo} ...")
        modelo = tf.keras.models.load_model(ruta_modelo)
        return modelo

    # Aquí solo entra si no existe archivo o si quieres reentrenar
    modelo, _ = entrenar_y_guardar_modelo(ruta_modelo=ruta_modelo)
    return modelo


if __name__ == "__main__":
    # Si ejecutas este archivo directamente: python entrenar_modelo.py
    # forzará el entrenamiento.
    cargar_o_entrenar_modelo(reentrenar=True)


Cargando datos MNIST...
Configurando aumento de datos...
Creando modelo...


c:\Users\Ximena\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Entrenando modelo...
Epoch 1/60
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 107s 56ms/step - accuracy: 0.5973 - loss: 1.1992 - val_accuracy: 0.9500 - val_loss: 0.1858
Epoch 2/60
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 107s 57ms/step - accuracy: 0.7772 - loss: 0.6890 - val_accuracy: 0.9662 - val_loss: 0.1148
Epoch 3/60
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 110s 58ms/step - accuracy: 0.8113 - loss: 0.5824 - val_accuracy: 0.9695 - val_loss: 0.1041
Epoch 4/60
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 98s 52ms/step - accuracy: 0.8302 - loss: 0.5275 - val_accuracy: 0.9736 - val_loss: 0.0836
Epoch 5/60
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 104s 55ms/step - accuracy: 0.8462 - loss: 0.4826 - val_accuracy: 0.9752 - val_loss: 0.0793
Epoch 6/60
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 137s 73ms/step - accuracy: 0.8540 - loss: 0.4531 - val_accuracy: 0.9739 - val_loss: 0.0840
Epoch 7/60
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 122s 65ms/step - accuracy: 0.8614 - loss: 0.4351 - val_accuracy: 0.9597 - val_loss: 0.1409
Epoch 8/60
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 113s 60

Guardando modelo en numeros_conv_ad_do.h5 ...
Modelo guardado correctamente.
